## Section 0 — Setup

In [ ]:
import os, gc, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import wandb

# ── Paths (Kaggle) ──────────────────────────────────────────────────────

OUT_DIR = "/kaggle/working"

ROLL_NO = "24f2003563"   # <-- CHANGE THIS to your actual roll number
WANDB_PROJECT = f"{ROLL_NO}-t22026"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [ ]:
def build_dataset(df, has_answer=True):
    """Expand each MCQ row into 5 rows (question + one candidate option each)."""
    rows = []
    for _, row in df.iterrows():
        for option in ['A', 'B', 'C', 'D', 'E']:
            text = f"Question: {row['prompt']} Candidate Answer: {row[option]}"
            item = {
                "id": row['id'],
                "prompt": row['prompt'],
                "option_text": row[option],
                "text": text,
                "option": option
            }
            if has_answer:
                item["label"] = 1 if option == row['answer'] else 0
            rows.append(item)
    return pd.DataFrame(rows)

train_processed = build_dataset(train, has_answer=True)
test_processed  = build_dataset(test,  has_answer=False)

print(train_processed.shape, test_processed.shape)
train_processed.head(10)


(10000, 6) (2500, 5)


,id,prompt,option_text,text,option,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Question: Pick the best possible answer: What ...,A,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,Question: Pick the best possible answer: What ...,B,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,Question: Pick the best possible answer: What ...,C,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,Question: Pick the best possible answer: What ...,D,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,Question: Pick the best possible answer: What ...,E,0
5,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,A,1
6,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,B,0
7,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,C,0
8,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,D,0
9,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,E,0


In [ ]:
# Train / Val split
train_ids, val_ids = train_test_split(train['id'].unique(), test_size=0.15, random_state=42)

train_df = train_processed[train_processed['id'].isin(train_ids)].reset_index(drop=True)
val_df   = train_processed[train_processed['id'].isin(val_ids)].reset_index(drop=True)

train_q = train[train['id'].isin(train_ids)].reset_index(drop=True)
val_q   = train[train['id'].isin(val_ids)].reset_index(drop=True)

print(f"Train questions: {len(train_ids)} | Val questions: {len(val_ids)}")


Train questions: 1700 | Val questions: 300


In [ ]:
# MAP@3 metric
def map_at_3(true_answers, pred_lists):
    """
    true_answers: list of correct option letters, e.g. ['A','C',...]
    pred_lists:   list of lists of top-3 predicted letters, e.g. [['A','B','C'], ...]
    """
    scores = []
    for true, preds in zip(true_answers, pred_lists):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

def scores_to_submission(id_list, option_list, score_list, out_path=None):
    """Convert per-option scores into a ranked top-3 submission dataframe."""
    df = pd.DataFrame({"id": id_list, "option": option_list, "score": score_list})
    rows = []
    for qid, g in df.groupby("id"):
        ranked = g.sort_values("score", ascending=False)
        top3 = ranked["option"].values[:3].tolist()
        rows.append({"ID": qid, "Prediction": " ".join(top3)})
    sub = pd.DataFrame(rows)
    if out_path:
        sub.to_csv(out_path, index=False)
    return sub


## W&B Login

In [ ]:
!wandb online

from google.colab import userdata
import wandb

api_key = userdata.get('WANDB_API_KEY')



wandb.login(key=api_key)

wandb.init(
    entity="24f2003563-iit-madras",
    project="24f2003563-t22026"
)

W&B online. Running your script from this directory will now sync to the cloud.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2003563 (24f2003563-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## TF-IDF / Word2Vec Baseline + Cosine Similarity


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df["text_clean"] = train_df["text"].apply(clean_text)
val_df["text_clean"]   = val_df["text"].apply(clean_text)
test_processed["text_clean"] = test_processed["text"].apply(clean_text)


In [ ]:
# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df["text_clean"])
X_val_tfidf   = tfidf.transform(val_df["text_clean"])
X_test_tfidf  = tfidf.transform(test_processed["text_clean"])

# ── Cosine similarity feature: prompt vs option text ────────────────────
prompt_vec_train = tfidf.transform(train_df["prompt"].apply(clean_text))
option_vec_train = tfidf.transform(train_df["option_text"].apply(clean_text))
cos_sim_train = np.array([
    cosine_similarity(prompt_vec_train[i], option_vec_train[i])[0, 0]
    for i in range(prompt_vec_train.shape[0])
])

prompt_vec_val = tfidf.transform(val_df["prompt"].apply(clean_text))
option_vec_val = tfidf.transform(val_df["option_text"].apply(clean_text))
cos_sim_val = np.array([
    cosine_similarity(prompt_vec_val[i], option_vec_val[i])[0, 0]
    for i in range(prompt_vec_val.shape[0])
])

prompt_vec_test = tfidf.transform(test_processed["prompt"].apply(clean_text))
option_vec_test = tfidf.transform(test_processed["option_text"].apply(clean_text))
cos_sim_test = np.array([
    cosine_similarity(prompt_vec_test[i], option_vec_test[i])[0, 0]
    for i in range(prompt_vec_test.shape[0])
])

print("Cosine sim feature ready. Example values:", cos_sim_train[:5])


Cosine sim feature ready. Example values: [0.1207969  0.13326118 0.2848606  0.34545644 0.10765669]


In [ ]:
from scipy.sparse import hstack, csr_matrix

X_train_full = hstack([X_train_tfidf, csr_matrix(cos_sim_train).T])
X_val_full   = hstack([X_val_tfidf,   csr_matrix(cos_sim_val).T])
X_test_full  = hstack([X_test_tfidf,  csr_matrix(cos_sim_test).T])

y_train = train_df["label"].values
y_val   = val_df["label"].values

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_full, y_train)

val_probs_tfidf = logreg.predict_proba(X_val_full)[:, 1]
val_preds_tfidf = (val_probs_tfidf > 0.5).astype(int)

acc_tfidf = accuracy_score(y_val, val_preds_tfidf)
f1_tfidf  = f1_score(y_val, val_preds_tfidf)
print(f"TF-IDF + LogReg — Val Accuracy: {acc_tfidf:.4f} | F1: {f1_tfidf:.4f}")


TF-IDF + LogReg — Val Accuracy: 0.9480 | F1: 0.8825


In [ ]:
# Compute MAP@3 for TF-IDF model
val_sub_tfidf = scores_to_submission(val_df["id"].values, val_df["option"].values, val_probs_tfidf)
val_answers = val_q.set_index("id").loc[val_sub_tfidf["ID"], "answer"].values
pred_lists_tfidf = val_sub_tfidf["Prediction"].str.split().tolist()

map3_tfidf = map_at_3(val_answers, pred_lists_tfidf)
print(f"TF-IDF + LogReg — Val MAP@3: {map3_tfidf:.4f}")


TF-IDF + LogReg — Val MAP@3: 0.9917


In [ ]:
# ── Log this run to W&B (Run 1 of 3+) ───────────────────────────────────
run = wandb.init(project=WANDB_PROJECT, name="m1-tfidf-logreg", config={
    "model": "TF-IDF + LogisticRegression",
    "max_features": 20000,
    "ngram_range": "(1,2)"
})
wandb.log({"val_accuracy": acc_tfidf, "val_f1": f1_tfidf, "val_map3": map3_tfidf})
run.finish()


val_accuracy,▁
val_f1,▁
val_map3,▁
val_accuracy,0.948
val_f1,0.88253
val_map3,0.99167


## Transformer Embeddings + Zero-Shot Classification

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))

def zero_shot_score(df):
    prompts = df["prompt"].tolist()
    options = df["option_text"].tolist()
    prompt_emb = embed_model.encode(prompts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    option_emb = embed_model.encode(options, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    # cosine similarity per row
    sims = np.sum(prompt_emb * option_emb, axis=1) / (
        np.linalg.norm(prompt_emb, axis=1) * np.linalg.norm(option_emb, axis=1) + 1e-8
    )
    return sims

val_scores_zeroshot = zero_shot_score(val_df)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
val_sub_zeroshot = scores_to_submission(val_df["id"].values, val_df["option"].values, val_scores_zeroshot)
pred_lists_zeroshot = val_sub_zeroshot["Prediction"].str.split().tolist()
map3_zeroshot = map_at_3(val_answers, pred_lists_zeroshot)
print(f"Zero-shot MiniLM embeddings — Val MAP@3: {map3_zeroshot:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m2-zeroshot-embeddings", config={
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "method": "zero-shot cosine similarity"
})
wandb.log({"val_map3": map3_zeroshot})
run.finish()


Zero-shot MiniLM embeddings — Val MAP@3: 0.4050


val_map3,▁
val_map3,0.405


## RAG Pipeline (Retrieval Augmentation)

In [ ]:
import faiss

# ── Build a simple vector DB from training prompts (swap for external corpus if available) ──
corpus_texts = train["prompt"].tolist()
corpus_emb = embed_model.encode(corpus_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
corpus_emb = corpus_emb / (np.linalg.norm(corpus_emb, axis=1, keepdims=True) + 1e-8)

index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb.astype("float32"))
print("FAISS index built with", index.ntotal, "vectors")


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

FAISS index built with 2000 vectors


In [ ]:
def retrieve_context(query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-8)
    scores, idxs = index.search(q_emb.astype("float32"), k)
    return [corpus_texts[i] for i in idxs[0]]

# Example retrieval
example_q = val_q.iloc[0]["prompt"]
print("Query:", example_q)
print("Retrieved context:", retrieve_context(example_q, k=2))


Query: Which of the following is correct? Which of the following statements accurately describes the blocking temperature of an antiferromagnetic layer in a spin valve? carefully.
Retrieved context: ['Which of the following is correct? Which of the following statements accurately describes the blocking temperature of an antiferromagnetic layer in a spin valve? carefully.', 'Which of the following is correct? Which of the following statements accurately describes the blocking temperature of an antiferromagnetic layer in a spin valve? carefully.']


In [ ]:
def build_rag_text(row, k=2):
    context = retrieve_context(row["prompt"], k=k)
    context_str = " | ".join(context)
    return f"Context: {context_str} Question: {row['prompt']} Candidate Answer: {row['option_text']}"

# Build RAG-augmented text for a validation subset (RAG retrieval is slow — sample if val set is large)
val_df_rag = val_df.copy()
val_df_rag["rag_text"] = val_df_rag.apply(lambda r: build_rag_text(r, k=2), axis=1)
val_df_rag[["text", "rag_text"]].head(3)


,text,rag_text
0,Question: Which of the following is correct? W...,Context: Which of the following is correct? Wh...
1,Question: Which of the following is correct? W...,Context: Which of the following is correct? Wh...
2,Question: Which of the following is correct? W...,Context: Which of the following is correct? Wh...


In [ ]:
# Score RAG-augmented text
rag_prompt_emb = embed_model.encode(val_df_rag["rag_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_option_emb = embed_model.encode(val_df_rag["option_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_sims = np.sum(rag_prompt_emb * rag_option_emb, axis=1) / (
    np.linalg.norm(rag_prompt_emb, axis=1) * np.linalg.norm(rag_option_emb, axis=1) + 1e-8
)

val_sub_rag = scores_to_submission(val_df_rag["id"].values, val_df_rag["option"].values, rag_sims)
pred_lists_rag = val_sub_rag["Prediction"].str.split().tolist()
map3_rag = map_at_3(val_answers, pred_lists_rag)
print(f"RAG-augmented zero-shot — Val MAP@3: {map3_rag:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m3-rag-pipeline", config={"k_retrieved": 2})
wandb.log({"val_map3": map3_rag})
run.finish()


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

RAG-augmented zero-shot — Val MAP@3: 0.4944


val_map3,▁
val_map3,0.49444


## LoRA Fine-Tuning (DeBERTa-v3)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType

model_name = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=384)


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [ ]:
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

class MCQDataset(TorchDataset):
    def __init__(self, hf_dataset):
        data = hf_dataset.to_dict()
        self.input_ids      = torch.tensor(data["input_ids"])
        self.attention_mask = torch.tensor(data["attention_mask"])
        self.labels         = torch.tensor(data["labels"]) if "labels" in data else None
        self.token_type_ids = torch.tensor(data["token_type_ids"]) if "token_type_ids" in data else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx]}
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

hf_train = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
hf_train = hf_train.rename_column("label", "labels")
hf_val   = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True)
hf_val   = hf_val.rename_column("label", "labels")

torch_train = MCQDataset(hf_train)
torch_val   = MCQDataset(hf_val)
print(f"Train rows: {len(torch_train)} | Val rows: {len(torch_val)}")


Map:   0%|          | 0/8500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Train rows: 8500 | Val rows: 1500


In [ ]:
# Class-weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        weight = torch.tensor([1.0, 4.0]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    pos_rate = preds.mean()   # sanity check: should land near 0.2
    return {"accuracy": acc, "f1": f1, "pred_positive_rate": pos_rate}


In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=64,
    lora_alpha=128,
    lora_dropout=0.15,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],  # DeBERTa-v3 attention projections
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()   # confirms only a tiny % of params are trainable


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

trainable params: 28,444,674 || all params: 463,508,484 || trainable%: 6.1368


In [ ]:
steps_per_epoch = max(1, len(torch_train) // (8 * 4))
total_steps = steps_per_epoch * 6
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/lora_outputs_v2",
    learning_rate=1.5e-4, 
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    weight_decay=0.02,
    max_grad_norm=1.0,
    warmup_steps=warmup_steps,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="m4-lora-r64-large",
)

trainer = WeightedTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=torch_train,
    eval_dataset=torch_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
eval_metrics_lora = trainer.evaluate()
print(eval_metrics_lora)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Pred Positive Rate
2.749894,0.695687,3,0.800000,0.000000,0.000000


{'eval_loss': 0.6956871747970581, 'eval_accuracy': 0.8, 'eval_f1': 0.0, 'eval_pred_positive_rate': 0.0}


In [ ]:
# Get LoRA model
lora_model.eval()
val_loader_simple = torch.utils.data.DataLoader(torch_val, batch_size=32)

all_probs_lora = []
with torch.no_grad():
    for batch in val_loader_simple:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        out = lora_model(**batch)
        probs = F.softmax(out.logits.float(), dim=-1)[:, 1]
        all_probs_lora.extend(probs.cpu().numpy())

val_sub_lora = scores_to_submission(val_df["id"].values, val_df["option"].values, np.array(all_probs_lora))
pred_lists_lora = val_sub_lora["Prediction"].str.split().tolist()
map3_lora = map_at_3(val_answers, pred_lists_lora)
print(f"LoRA DeBERTa-v3 — Val MAP@3: {map3_lora:.4f}")

wandb.init(project=WANDB_PROJECT, name="m4-lora-final-map3", config={"model": "deberta-v3-large+LoRA"})
wandb.log({"val_map3": map3_lora, **eval_metrics_lora})
wandb.finish()


LoRA DeBERTa-v3 — Val MAP@3: 0.3417


eval/accuracy,▁▁▁▁
eval/f1,▁▁▁▁
eval/loss,▂█▁▂
eval/pred_positive_rate,▁▁▁▁
eval/runtime,▁▃▄█
eval/samples_per_second,█▆▅▁
eval/steps_per_second,█▆▃▁
train/epoch,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇████
train/global_step,▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇████
train/grad_norm,▂▂▁▇▇▃█▂▂▁▅▂▁▇▅
+2,...


eval_accuracy,▁
eval_f1,▁
eval_loss,▁
eval_pred_positive_rate,▁
val_map3,▁
eval_accuracy,0.8
eval_f1,0
eval_loss,0.69569
eval_pred_positive_rate,0
val_map3,0.34167


In [ ]:
# Build RAG-augmented text
test_processed_rag = test_processed.copy()
test_processed_rag["rag_text"] = test_processed_rag.apply(lambda r: build_rag_text(r, k=2), axis=1)

# ── Score RAG-augmented text using the same zero-shot embedding approach ──
rag_prompt_emb_test = embed_model.encode(test_processed_rag["rag_text"].tolist(), batch_size=32, show_progress_bar=True)
rag_option_emb_test = embed_model.encode(test_processed_rag["option_text"].tolist(), batch_size=32, show_progress_bar=True)

test_rag_scores = np.sum(rag_prompt_emb_test * rag_option_emb_test, axis=1) / (
    np.linalg.norm(rag_prompt_emb_test, axis=1) * np.linalg.norm(rag_option_emb_test, axis=1) + 1e-8
)

print("test_rag_scores ready:", test_rag_scores.shape)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

test_rag_scores ready: (2500,)


## Milestone 5: Ensembling


In [ ]:
def normalize(scores):
    scores = np.array(scores, dtype=float)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

ensemble_scores = (
    0.2 * normalize(val_probs_tfidf) +
    0.2 * normalize(val_scores_zeroshot) +
    0.6 * normalize(all_probs_lora)        # weight the strongest model highest
)

val_sub_ensemble = scores_to_submission(val_df["id"].values, val_df["option"].values, ensemble_scores)
pred_lists_ensemble = val_sub_ensemble["Prediction"].str.split().tolist()
map3_ensemble = map_at_3(val_answers, pred_lists_ensemble)
print(f"Ensemble (TFIDF + ZeroShot + LoRA) — Val MAP@3: {map3_ensemble:.4f}")

run = wandb.init(project=WANDB_PROJECT, name="m5-ensemble", config={
    "weights": {"tfidf": 0.2, "zeroshot": 0.2, "lora": 0.6}
})
wandb.log({"val_map3": map3_ensemble})
run.finish()


Ensemble (TFIDF + ZeroShot + LoRA) — Val MAP@3: 0.8333


val_map3,▁
val_map3,0.83333


In [ ]:
# ── Summary table comparing all runs (for your report) ──────────────────
summary = pd.DataFrame([
    {"model": "TF-IDF + LogisticRegression", "val_map3": map3_tfidf, "val_f1": f1_tfidf, "val_acc": acc_tfidf},
    {"model": "Zero-shot MiniLM embeddings", "val_map3": map3_zeroshot},
    {"model": "RAG-augmented embeddings",    "val_map3": map3_rag},
    {"model": "LoRA fine-tuned DeBERTa-v3",  "val_map3": map3_lora, **eval_metrics_lora},
    {"model": "Ensemble (all 3 main models)","val_map3": map3_ensemble},
])
summary.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)
summary


,model,val_map3,val_f1,val_acc,eval_loss,eval_accuracy,eval_f1,eval_pred_positive_rate
0,TF-IDF + LogisticRegression,0.991667,0.88253,0.948,NaN,NaN,NaN,NaN
1,Zero-shot MiniLM embeddings,0.405000,NaN,NaN,NaN,NaN,NaN,NaN
2,RAG-augmented embeddings,0.494444,NaN,NaN,NaN,NaN,NaN,NaN
3,LoRA fine-tuned DeBERTa-v3,0.341667,NaN,NaN,0.695687,0.8,0.0,0.0
4,Ensemble (all 3 main models),0.833333,NaN,NaN,NaN,NaN,NaN,NaN
